# ZenteiQ AI Tech Innovations — Task 2: Dense Model (Qwen)
## MaxText Qwen3-1.7B CPU Training Run & Benchmark

This notebook provides a detailed setup and execution workflow for training the **Qwen3-1.7B** base model on a **CPU-only** environment using **MaxText** and **JAX** with synthetic data.

### 1. Installation & Environment Setup

We will clone the MaxText repository and install its dependencies. When running on a CPU-only environment (such as Google Colab with the CPU runtime selected), JAX defaults to CPU execution. We will install the dependencies using `uv` for fast installations.

In [2]:
# Clone the MaxText repository
!git clone https://github.com/AI-Hypercomputer/maxtext.git
%cd maxtext

# Install uv (fast package installer)
!pip install uv

# Set environmental variables to avoid GPU/TPU requirements
import os
os.environ["UV_TORCH_BACKEND"] = "cpu"

# Install MaxText CPU dependencies
!uv pip install -e .

# Restart the runtime after this step if run in Colab!

Cloning into 'maxtext'...
remote: Enumerating objects: 97091, done.
remote: Counting objects: 100% (2350/2350), done.
remote: Compressing objects: 100% (1184/1184), done.
remote: Total 97091 (delta 1906), reused 1206 (delta 1166), pack-reused 94741 (from 3)
Receiving objects: 100% (97091/97091), 411.51 MiB | 19.34 MiB/s, done.
Resolving deltas: 100% (70834/70834), done.
/content/maxtext
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.1/25.1 MB 50.1 MB/s eta 0:00:00:00:0100:01
Using Python 3.12.13 environment at: /usr
Resolved 1 package in 791ms                                          
Prepared 1 package in 1ms                                                
Installed 1 package in 2msm file:///content/maxtext)        
 + maxtext==0.2.3 (from file:///content/maxtext)


In [9]:
!uv pip install qwix -q

In [10]:
!uv pip install tokamax -q

### 2. Verify JAX CPU Backend Connection

Run the following cell to confirm that JAX successfully detects the CPU backend.

In [3]:
!ls

AUTHORS      CONTRIBUTING.md  LICENSE_HEADER  pyproject.toml  scripts  tools
benchmarks   docs	      PREFLIGHT.md    pytest.ini      src
codecov.yml  LICENSE	      pylintrc	      README.md       tests


In [3]:
!uv pip install aqtp

Using Python 3.12.13 environment at: /usr
Resolved 30 packages in 264ms                                        
Prepared 1 package in 101ms                                              
Installed 1 package in 10ms                                 
 + aqtp==0.9.0


In [2]:
import jax
print("JAX Version:", jax.__version__)
print("Available devices:", jax.devices())
print("Default backend:", jax.default_backend())

JAX Version: 0.10.2
Available devices: [CpuDevice(id=0)]
Default backend: cpu


### 3. Configuration Breakdown

Below are the parameters we are setting for this training run, along with their purpose and effect:

| Configuration Parameter | Value | Purpose | Effect |
| :--- | :--- | :--- | :--- |
| **`model_name`** | `qwen3-0.6b` | Specifies the base architecture layout to use. | Loads structural config from `configs/models/qwen3-0.6b.yml` setting embedding dimensions (`base_emb_dim: 1024`), intermediate MLP dimension (`base_mlp_dim: 3072`), attention query/KV heads (`16`/`8`), and number of layers (`28`). |
| **`steps`** | `50` | Defines training duration. | Runs the optimization/training loop for exactly 50 iterations, which is sufficient to compile the graph, calculate device step times, and measure TFLOPs throughput. |
| **`dataset_type`** | `synthetic` | Determines the data loader pipeline format. | Bypasses external dataset downloading and disk I/O bottlenecks by dynamically generating random tensors of input tokens on CPU memory. This allows pure computational throughput benchmarking. |
| **`base_output_directory`** | `./output` | Sets the file storage root path. | All metrics, TensorBoard execution logs, compile metadata, and model checkpoints are saved under this root directory. |
| **`run_name`** | `qwen3-0.6b-cpu` | Uniquely names the specific execution run. | Prevents overwriting other runs and creates nested folders named `qwen3-0.6b-cpu/` for output checkpoints and statistics. |

### 4. Execute CPU Training (Qwen 1.7B)

We run the main training script with our configuration overrides. We capture the stdout logs to parse metrics later.

In [5]:
!cat src/maxtext/configs/base.yml

# Copyright 2023–2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#    https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# This sentinel is a reminder to choose a real run name.
# If there is already a checkpoint under this run, that checkpoint will auto-resume.
#
# NOTE: Some unit/integration tests in MaxText do not always run this file directly.
# When running in decoupled mode (DECOUPLE_GCLOUD=TRUE), tests may use
# `decoupled_base_test.yml` instead of `base.yml` via `tests/utils/test_helpers.py`.
run_name: ""

model_name: "default"

In [15]:
# # Run training for 50 steps on CPU
# !python3 -m maxtext.trainers.pre_train.train src/maxtext/configs/base.yml \
#     model_name=qwen3-0.6b \
#     steps=50 \
#     dataset_type=synthetic \
#     base_output_directory=./cpu-output \
#     run_name=qwen3-0.6b-cpu \
#     per_device_batch_size=1 \
#     attention=dot_product \
#     max_target_length=256 \
#     opt_type=sgd \
#     enable_checkpointing=false | tee train_cpu_run.log

In [13]:
!sudo apt-get update && sudo apt-get install -y \
    libxcomposite1 \
    libxcursor1 \
    libxdamage1 \
    libxext6 \
    libxfixes3 \
    libxi6 \
    libxrandr2 \
    libxrender1 \
    libxtst6 \
    libgbm1 \
    libasound2



Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease    
Get:9 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,358 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,612 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,489 kB]
Get:13 https://r2u.stat.illinois.ed

In [27]:
!ls maxtext/src/maxtext/

assets		       eval	     __init__.py     layers	 __pycache__
checkpoint_conversion  examples      input_pipeline  models	 scratch_code
common		       experimental  integration     multimodal  trainers
configs		       inference     kernels	     optimizers  utils


In [28]:
!grep -E "jax_distributed|coordinator|base_num_decoder_layers|num_decoder_layers" maxtext/src/maxtext/configs/base.yml

# base_mlp_dim, base_num_decoder_layers and/or head_dim.
base_num_decoder_layers: 16
# The number of repeats will be set to num_decoder_layers / (num_pipeline_stages * num_layers_per_pipeline_stage)
jax_distributed_initialization_timeout: 300 # This is the default timeout in https://github.com/jax-ml/jax/blob/main/jax/_src/distributed.py
skip_jax_distributed_system: false # If true we will not initialize the jax distributed system.


In [29]:
# Run training for 50 steps on CPU
!python3 -m maxtext.trainers.pre_train.train src/maxtext/configs/base.yml \
    model_name=qwen3-1.7b \
    steps=50 \
    dataset_type=synthetic \
    base_output_directory=$(pwd)/cpu-output-1.7b \
    run_name=qwen3-1.7b-cpu\
    override_model_config=true \
    skip_jax_distributed_system=true \
    base_emb_dim=512 \
    base_mlp_dim=1536 \
    base_num_decoder_layers=8 \
    base_num_query_heads=8 \
    base_num_kv_heads=4 \
    per_device_batch_size=1 \
    attention=dot_product \
    # opt_type=sgd \
    max_target_length=512 \
    weight_dtype=bfloat16 \
    grad_dtype=bfloat16 \
    enable_checkpointing=false

[transformers] DeepseekV32Config got `key=rope_scaling` in kwargs but hasn't set it as attribute. For RoPE standardization you need to set `self.rope_parameters` in model's config. 
[NO-OP] mldiagnostics: dependency missing; using stub. (ModuleNotFoundError)
[NO-OP] mldiagnostics: dependency missing; using stub. (ModuleNotFoundError)
[NO-OP] mldiagnostics: dependency missing; using stub. (ModuleNotFoundError)
[NO-OP] vertex_tensorboard: dependency missing; using stub. (ModuleNotFoundError)
2026-06-18 09:43:40.846533: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
W0618 09:43:41.175636 134761795641344 pyconfig.py:294] tokenizer_path not found in HF_IDS in maxtext/src/maxtext/utils/globals.py.           Using the default src/maxtext/assets/tokenizers/tokenizer.llama2 instead.           Please pass tokenizer_path in your command if this is not intended.
I0618 09:43:41.176163 1347617

In [5]:
!uv pip install aqtp ml-goodput-measurement ml-collections

Using Python 3.12.13 environment at: /usr
Resolved 56 packages in 519ms                                        
Prepared 3 packages in 23ms                                              
Uninstalled 1 package in 2ms
Installed 3 packages in 5msnt==0.2.0                        
 - grpcio-status==1.71.2
 + grpcio-status==1.81.0
 + ml-collections==1.1.0
 + ml-goodput-measurement==0.2.0


In [17]:
!uv pip install pathwaysutils aqt

Using Python 3.12.13 environment at: /usr
Checked 2 packages in 95ms


In [18]:
!uv pip install -U transformers

Using Python 3.12.13 environment at: /usr
Resolved 27 packages in 264ms                                        
Prepared 10 packages in 1.00s                                            
Uninstalled 10 packages in 773ms
Installed 10 packages in 121ms                              
 - anyio==4.13.0
 + anyio==4.14.0
 - certifi==2026.5.20
 + certifi==2026.6.17
 - filelock==3.29.2
 + filelock==3.29.4
 - fsspec==2025.3.0
 + fsspec==2026.6.0
 - huggingface-hub==1.18.0
 + huggingface-hub==1.19.0
 - numpy==2.0.2
 + numpy==2.4.6
 - regex==2025.11.3
 + regex==2026.5.9
 - rich==13.9.4
 + rich==15.0.0
 - tqdm==4.67.3
 + tqdm==4.68.3
 - transformers==5.10.2
 + transformers==5.12.1


In [15]:
!uv pip install drjax -q

In [33]:
!ls cpu-output-1.7b/qwen3-1.7b-cpu/checkpoints/

0  49


### 5. Parse Metrics & Logs

MaxText prints metric logs regularly during training. We will extract the compilation time, step times, loss, learning rate, and TFLOPs using a helper script.

In [40]:
%load_ext tensorboard


In [1]:
%tensorboard --logdir cpu-output/qwen3-0.6b-cpu/tensorboard/


UsageError: Line magic function `%tensorboard` not found.


In [34]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!ls drive/

MyDrive  Othercomputers


In [35]:
!cp -r cpu-output-1.7b /content/drive/MyDrive/
